In [1]:
%pip install duckdb faker

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 12.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import duckdb
import random

from faker import Faker
from datetime import datetime
from decimal import Decimal
from pprint import pprint

con = duckdb.connect("ecommerce.duckdb")

fake = Faker("pt_BR")

print("Conexão com DuckDB estabelecida!")

Conexão com DuckDB estabelecida!


In [2]:
con.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id UUID PRIMARY KEY DEFAULT uuid(),
    nome VARCHAR NOT NULL,
    email VARCHAR NOT NULL,
    endereco TEXT,
    telefone VARCHAR
);
""")

con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
    id UUID PRIMARY KEY DEFAULT uuid(),
    nome VARCHAR NOT NULL,
    descricao TEXT,
    preco DECIMAL(10,2) NOT NULL,
    categoria VARCHAR,
    estoque INTEGER NOT NULL
);
""")

con.execute("""
CREATE TABLE IF NOT EXISTS pedidos (
    id UUID PRIMARY KEY DEFAULT uuid(),
    cliente_id UUID NOT NULL REFERENCES clientes(id),
    produto_id UUID NOT NULL REFERENCES produtos(id),
    quantidade INTEGER NOT NULL,
    data_pedido TIMESTAMP NOT NULL
);
""")

print("Tabelas criadas com sucesso!")

Tabelas criadas com sucesso!


In [3]:
categorias = [
    "Eletrônicos",
    "Roupas",
    "Livros",
    "Alimentos",
    "Móveis"
]

def inserir_dados(
    num_clientes=100,
    num_produtos=50,
    num_pedidos=200
):
    clientes = []

    for _ in range(num_clientes):
        clientes.append((
            fake.name(),
            fake.email(),
            fake.address(),
            fake.phone_number()
        ))

    produtos = []

    for _ in range(num_produtos):
        produtos.append((
            fake.word().capitalize(),
            fake.sentence(),
            Decimal(random.randint(1000, 100000)) / 100,
            random.choice(categorias),
            random.randint(1, 100)
        ))

    try:
        con.begin()

        if clientes:
            con.executemany("""
                INSERT INTO clientes
                (nome, email, endereco, telefone)
                VALUES (?, ?, ?, ?)
            """, clientes)

        if produtos:
            con.executemany("""
                INSERT INTO produtos
                (nome, descricao, preco, categoria, estoque)
                VALUES (?, ?, ?, ?, ?)
            """, produtos)

        clientes_ids = [
            row[0] for row in con.execute(
                "SELECT id FROM clientes"
            ).fetchall()
        ]

        produtos_ids = [
            row[0] for row in con.execute(
                "SELECT id FROM produtos"
            ).fetchall()
        ]

        pedidos = []

        for _ in range(num_pedidos):
            data_pedido = datetime.combine(
                fake.date_this_year(),
                datetime.min.time()
            )

            pedidos.append((
                random.choice(clientes_ids),
                random.choice(produtos_ids),
                random.randint(1, 5),
                data_pedido
            ))

        if pedidos:
            con.executemany("""
                INSERT INTO pedidos
                (cliente_id, produto_id, quantidade, data_pedido)
                VALUES (?, ?, ?, ?)
            """, pedidos)

        con.commit()

        print(f"{num_clientes} clientes inseridos!")
        print(f"{num_produtos} produtos inseridos!")
        print(f"{num_pedidos} pedidos inseridos!")

    except Exception:
        con.rollback()
        raise

In [4]:
inserir_dados(
    num_clientes=100,
    num_produtos=50,
    num_pedidos=200
)

100 clientes inseridos!
50 produtos inseridos!
200 pedidos inseridos!


In [5]:
inserir_dados(
    num_clientes=100,
    num_produtos=50,
    num_pedidos=200
)

100 clientes inseridos!
50 produtos inseridos!
200 pedidos inseridos!


In [6]:
print("=== Contagem de Registros ===")

for tabela in ["clientes", "produtos", "pedidos"]:
    total = con.execute(
        f"SELECT COUNT(*) FROM {tabela}"
    ).fetchone()[0]

    print(f"{tabela}: {total}")


print("\n=== Exemplo de Cliente ===")
pprint(
    con.execute(
        "SELECT * FROM clientes LIMIT 1"
    ).fetchone()
)

print("\n=== Exemplo de Produto ===")
pprint(
    con.execute(
        "SELECT * FROM produtos LIMIT 1"
    ).fetchone()
)

print("\n=== Exemplo de Pedido ===")
pprint(
    con.execute(
        "SELECT * FROM pedidos LIMIT 1"
    ).fetchone()
)

=== Contagem de Registros ===
clientes: 200
produtos: 100
pedidos: 400

=== Exemplo de Cliente ===
(UUID('9df7a5b4-167b-4d6b-a849-4967091d590c'),
 'Anna Liz Rodrigues',
 'leo20@example.org',
 'Passarela de Oliveira, 34\n'
 'São Francisco Das Chagas\n'
 '65076-753 Moura dos Dourados / AC',
 '+55 34 6630-8775')

=== Exemplo de Produto ===
(UUID('f774978a-4e5a-4673-ae6c-2b1e02b36e51'),
 'Cum',
 'Cum enim asperiores aut.',
 Decimal('326.77'),
 'Alimentos',
 60)

=== Exemplo de Pedido ===
(UUID('8b890ac6-c5ad-4375-a4d0-a9c3d29783ec'),
 UUID('013aa251-fc8e-48e0-aee6-e9cd829b68c3'),
 UUID('190d563f-d102-4231-a338-3b191d50aa03'),
 3,
 datetime.datetime(2026, 1, 3, 0, 0))


In [7]:
produto_mais_caro = con.execute("""
    SELECT nome, preco
    FROM produtos
    ORDER BY preco DESC
    LIMIT 1
""").fetchone()

print(
    f"Produto mais caro: {produto_mais_caro[0]} "
    f"- R$ {produto_mais_caro[1]}"
)

Produto mais caro: Enim - R$ 996.23


In [10]:
total_estoque = con.execute("""
    SELECT COALESCE(SUM(estoque), 0)
    FROM produtos
""").fetchone()[0]

print(f"Total em estoque: {total_estoque}")

Total em estoque: 4792


In [ ]:
pedidos_por_categoria = con.execute("""
    SELECT
        p.categoria,
        COUNT(pe.id) AS total_pedidos
    FROM pedidos pe
    JOIN produtos p
        ON pe.produto_id = p.id
    GROUP BY p.categoria
    ORDER BY total_pedidos DESC
""").fetchall()

for categoria, total in pedidos_por_categoria:
    print(f"{cpedidos_por_categoria = con.execute("""
    SELECT
        p.categoria,
        COUNT(pe.id) AS total_pedidos
    FROM pedidos pe
    JOIN produtos p
        ON pe.produto_id = p.id
    GROUP BY p.categoria
    ORDER BY total_pedidos DESC
""").fetchall()

for categoria, total in pedidos_por_categoria:
    print(f"{categoria}: {total}")ategoria}: {total}")

Móveis: 117
Eletrônicos: 74
Alimentos: 73
Roupas: 71
Livros: 65


In [13]:
inserir_dados(
    num_clientes=200,
    num_produtos=150,
    num_pedidos=150
)

200 clientes inseridos!
150 produtos inseridos!
150 pedidos inseridos!


In [14]:
resultado = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM clientes) AS clientes,
        (SELECT COUNT(*) FROM produtos) AS produtos,
        (SELECT COUNT(*) FROM pedidos) AS pedidos
""").fetchone()

print(f"Clientes: {resultado[0]}")
print(f"Produtos: {resultado[1]}")
print(f"Pedidos: {resultado[2]}")

Clientes: 400
Produtos: 250
Pedidos: 550


In [15]:
df_clientes = con.execute("""
    SELECT * FROM clientes
""").df()

df_produtos = con.execute("""
    SELECT * FROM produtos
""").df()

df_pedidos = con.execute("""
    SELECT * FROM pedidos
""").df()

display(df_clientes.head())
display(df_produtos.head())
display(df_pedidos.head())

,id,nome,email,endereco,telefone
0,9df7a5b4-167b-4d6b-a849-4967091d590c,Anna Liz Rodrigues,leo20@example.org,"Passarela de Oliveira, 34\nSão Francisco Das C...",+55 34 6630-8775
1,87fd3fa3-ffd6-47fb-ab90-ad284f91d749,Anthony Pastor,pereiraana-beatriz@example.org,"Área Viana, 91\nJardim Felicidade\n29071-906 M...",15 0429-8386
2,666f7752-9b9f-4f18-83f0-e714d8de6d2b,Maria Julia da Paz,murilo56@example.org,"Lago de Souza, 10\nCanadá\n24390-254 Brito / MS",+55 43 3952-2423
3,e857ea48-1810-4f77-921c-61e9c2229c98,Ana Cecília Pastor,raquelcardoso@example.org,"Viaduto Moraes, 346\nPrimeiro De Maio\n34561-9...",+55 15 1394-2970
4,a2061022-7cd0-4afc-8510-72c13d66554b,Matheus Moreira,das-nevesnoah@example.com,"Lagoa Caleb Pinto, 83\nVale Do Jatoba\n2223926...",+55 12 7879-8038


,id,nome,descricao,preco,categoria,estoque
0,f774978a-4e5a-4673-ae6c-2b1e02b36e51,Cum,Cum enim asperiores aut.,326.77,Alimentos,60
1,27f40cbe-824b-4a87-a5cc-e272546c12e3,Fugiat,Ducimus delectus tempore maiores culpa laborio...,714.20,Móveis,44
2,98333daa-cd36-4e27-bdaa-be58623ee4cc,Enim,Corrupti impedit deleniti.,996.23,Livros,63
3,5aa91819-f988-4146-bc5f-cb6b1779decf,Cum,A voluptatum esse cupiditate reiciendis volupt...,566.36,Móveis,85
4,c52f5dd8-778d-49ed-845b-633fa2debd68,Sit,Veritatis nam ratione ratione facere.,120.58,Móveis,25


,id,cliente_id,produto_id,quantidade,data_pedido
0,8b890ac6-c5ad-4375-a4d0-a9c3d29783ec,013aa251-fc8e-48e0-aee6-e9cd829b68c3,190d563f-d102-4231-a338-3b191d50aa03,3,2026-01-03
1,5c68bc4c-a659-4cda-a19b-062bd0c42ab3,74c032f5-7e1c-427a-ab8a-7aa65b8ae706,4e855402-eb72-43f0-b1b0-8230cd236934,2,2026-03-27
2,bc0cd43f-07c1-48e7-abe5-c0fa779765ab,f18dbe43-efec-43ef-9b91-0a2ee4d2b14c,98333daa-cd36-4e27-bdaa-be58623ee4cc,1,2026-05-15
3,58d48629-9f17-45c9-b076-11477f7698b3,4977fb7b-ecc2-40e1-929d-49aae3dc65ee,762d6d2a-f7bb-411a-8be5-66acda21464a,1,2026-06-02
4,8445b8c0-fd5b-4f6f-a24b-0e86f50ac5bc,2dffed02-4af1-43a3-8c85-400f962074a4,7354629c-5621-41dd-a593-167ff04af676,4,2026-08-04


In [16]:
con.close()